In [ ]:
import torch
import vitlab
from vitlab.activations import ActivationReader
from vitlab.datasets import get_splits
from vitlab.sae import load_layer_sae

DEVICE = "cuda"
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/dinov2_fitz_l9_topk16_d3072"
DATASET = "fitzpatrick17k"

model  = vitlab.load_model(BACKBONE_CKPT, device=DEVICE)
reader = ActivationReader(model.backbone)
sae = load_layer_sae(SAE_CKPT, device=DEVICE)

train, _, _ = get_splits(DATASET, model_key=model.spec.key)
px = train[0]["pixel_values"].unsqueeze(0).to(DEVICE)

In [ ]:
with torch.no_grad():
    acts    = reader.read(px, sae.spec.site)
    print(f"Activations Shape (Batch, Sequence, Dimension): {acts.shape}")
    patches = acts[:, model.spec.n_prefix_tokens:, :]
    print(f"Patches Shape (Batch, Patches, Dimension): {patches.shape}")
    codes   = sae.encode(patches.reshape(-1, patches.shape[-1]))
    print(f"Codes Shape (Batch x Sequence, SAE Dimension): {codes.shape}")

active = (codes > 0).sum(-1).float().mean()
print(f"{codes.shape[1]} concepts, {active:.1f} active per patch (should be ~top_k)")

In [ ]:
sae.spec

In [ ]:
import torch
import vitlab
from vitlab.datasets import get_splits
from vitlab.sae import load_bank                 # -> SAEBank keyed by site
from vitlab import viz
import vitlab.attribution as A

device = "cuda"
SITE = "blocks.9.resid_post"          # must match what the SAE was trained on
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/dinov2_fitz_l9_topk16_d3072"
DATASET = "fitzpatrick17k"

# 1. model (backbone + task head) and the SAE bank
model = vitlab.load_model(BACKBONE_CKPT, device=device)
bank  = load_bank({SITE: SAE_CKPT}, device=device)   # {site: sae_dir}

# 2. one image, preprocessed for this backbone; keep the batch dim
train, _, _ = get_splits(DATASET, model_key=model.spec.key)
image = train[0]["pixel_values"].unsqueeze(0)      # (1, 3, 224, 224)

# 3. two-stage attribution: patching screens -> ablation verifies
task = model.task_names[0]                         # or e.g. "fitzpatrick17k"


In [ ]:
image = train[0]["pixel_values"].unsqueeze(0)      # (1, 3, 224, 224)
r = A.attribution_patching(model, bank, image, SITE, task=task, top_k=10, device=device)

for c in r.top(5):
    print(f"#{c['rank']}  {c['site']}  F{c['feature']}  Δ={c['score']:.4f}")

# 4. figure
viz.use_style()
fig = viz.attribution_grid(image, r, model_key=model.spec.key, save="attr.pdf", show=True)

In [ ]:
_ = viz.attribution_stats(r, title="Attr Stats", show=True)

In [ ]:
share = A.token_group_attribution(model, image, task=task)
print(share.table())
_ = viz.token_group_trajectory(share, measure="ablation_drop", save="tg.pdf")

In [ ]:
import vitlab
import torch, warnings
from torch.utils.data import DataLoader, TensorDataset
from vitlab.sae import discover_bank
from vitlab import get_splits, make_loader 
from vitlab.circuits import (AttributionPatcher, CircuitDiscovery, compute_median_activations,
                             verify_edges, CircuitEvaluator, collect_class_images)
import vitlab.viz as V
V.use_style()

device = "cuda"
BACKBONE_CKPT = "../runs/dinov2-base_full_epochs30/best"
SAE_CKPT = "../saes/"
DATASET = "fitzpatrick17k"
TARGET_CLASS = 8

# 1. model (backbone + heads) and a bank with an SAE at EVERY layer
model = vitlab.load_model(BACKBONE_CKPT, device=device)
n_layers = model.spec.n_layers
bank = discover_bank(SAE_CKPT, device=device)   # walks the tree, keys by site
task = model.task_names[0]                                 # or task=DATASET explicitly

# 2. data: one loader for the median baseline (balanced), and class images to explain
train, _, _ = get_splits(DATASET, model_key=model.spec.key)
ref_loader = make_loader(train, batch_size=32, shuffle=True, num_workers=4)
class_images = collect_class_images(ref_loader, TARGET_CLASS, n_images=32, device=device)


In [ ]:
# 1. median baseline + discovery
medians = compute_median_activations(model, bank, ref_loader, n_layers, device=device, max_batches=20)
patcher = AttributionPatcher(model, bank, medians, task=task, device=device)
disco   = CircuitDiscovery(patcher, n_layers)               # layer set derived from bank
circ    = disco.discover_aggregated(class_images, TARGET_CLASS, top_k=5, use_libragrad=True)
circ.summary()
_ = verify_edges(model, bank, circ, class_images[0], task=task, device=device)
_ = circ.plot(save="circuit.pdf", show=True)



In [ ]:


# 3. evaluation: single circuit + AUC-over-k sweep
ev = CircuitEvaluator(model, bank, medians, target_class=TARGET_CLASS,
                      n_layers=n_layers, task=task, device=device)
res = ev.evaluate(class_images[0], circ)
print(f"faithfulness={res.faithfulness:.3f}  completeness={res.completeness:.3f}  causality={res.causality:.3f}")
results, auc_f, auc_c = ev.evaluate_over_k(class_images, disco)

# 4. all figures


#

In [ ]:
V.circuit_viz.circuit_metrics_curve(results, auc_faith=auc_f, auc_comp=auc_c,
                        max_features=1536, save="circuit_metrics.pdf")

In [ ]:
circ.to_html(model, bank, ref_loader, "circuit.html", model_key="dinov2-base")

In [ ]:
train.features["label"]

In [24]:
import pandas as pd
import os
datasets = ["7ptderm", "fitzpatrick/skincon", "CUB_200_2011"]
splits = ["train", "val", "test"]

meta_file = "metadata.csv"
filename_column = {
    "7ptderm": "file_name",
    "fitzpatrick/skincon": "file_name",
    "CUB_200_2011": "file_name"
}
concept_columns = {
    "7ptderm": ["pigment_network", 'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules',
       'blue_whitish_veil', 'vascular_structures', "elevation"],
    "fitzpatrick/skincon": ['has_Vesicle', 'has_Papule', 'has_Macule', 'has_Plaque', 'has_Abscess',
       'has_Pustule', 'has_Bulla', 'has_Patch', 'has_Nodule', 'has_Ulcer',
       'has_Crust', 'has_Erosion', 'has_Excoriation', 'has_Atrophy',
       'has_Exudate', 'has_Purpura/Petechiae', 'has_Fissure', 'has_Induration',
       'has_Xerosis', 'has_Telangiectasia', 'has_Scale', 'has_Scar',
       'has_Friable', 'has_Sclerosis', 'has_Pedunculated',
       'has_Exophytic/Fungating', 'has_Warty/Papillomatous', 'has_Dome-shaped',
       'has_Flat topped', 'has_Brown(Hyperpigmentation)', 'has_Translucent',
       'has_White(Hypopigmentation)', 'has_Purple', 'has_Yellow', 'has_Black',
       'has_Erythema', 'has_Comedo', 'has_Lichenification', 'has_Blue',
       'has_Umbilicated', 'has_Poikiloderma', 'has_Salmon', 'has_Wheal',
       'has_Acuminate', 'has_Burrow', 'has_Gray', 'has_Pigmented', 'has_Cyst'],
    "CUB_200_2011": ['has_back_color::black',
 'has_back_color::blue',
 'has_back_color::brown',
 'has_back_color::buff',
 'has_back_color::green',
 'has_back_color::grey',
 'has_back_color::iridescent',
 'has_back_color::olive',
 'has_back_color::orange',
 'has_back_color::pink',
 'has_back_color::purple',
 'has_back_color::red',
 'has_back_color::rufous',
 'has_back_color::white',
 'has_back_color::yellow',
 'has_back_pattern::multi-colored',
 'has_back_pattern::solid',
 'has_back_pattern::spotted',
 'has_back_pattern::striped',
 'has_belly_color::black',
 'has_belly_color::blue',
 'has_belly_color::brown',
 'has_belly_color::buff',
 'has_belly_color::green',
 'has_belly_color::grey',
 'has_belly_color::iridescent',
 'has_belly_color::olive',
 'has_belly_color::orange',
 'has_belly_color::pink',
 'has_belly_color::purple',
 'has_belly_color::red',
 'has_belly_color::rufous',
 'has_belly_color::white',
 'has_belly_color::yellow',
 'has_belly_pattern::multi-colored',
 'has_belly_pattern::solid',
 'has_belly_pattern::spotted',
 'has_belly_pattern::striped',
 'has_bill_color::black',
 'has_bill_color::blue',
 'has_bill_color::brown',
 'has_bill_color::buff',
 'has_bill_color::green',
 'has_bill_color::grey',
 'has_bill_color::iridescent',
 'has_bill_color::olive',
 'has_bill_color::orange',
 'has_bill_color::pink',
 'has_bill_color::purple',
 'has_bill_color::red',
 'has_bill_color::rufous',
 'has_bill_color::white',
 'has_bill_color::yellow',
 'has_bill_length::about_the_same_as_head',
 'has_bill_length::longer_than_head',
 'has_bill_length::shorter_than_head',
 'has_bill_shape::all-purpose',
 'has_bill_shape::cone',
 'has_bill_shape::curved_(up_or_down)',
 'has_bill_shape::dagger',
 'has_bill_shape::hooked',
 'has_bill_shape::hooked_seabird',
 'has_bill_shape::needle',
 'has_bill_shape::spatulate',
 'has_bill_shape::specialized',
 'has_breast_color::black',
 'has_breast_color::blue',
 'has_breast_color::brown',
 'has_breast_color::buff',
 'has_breast_color::green',
 'has_breast_color::grey',
 'has_breast_color::iridescent',
 'has_breast_color::olive',
 'has_breast_color::orange',
 'has_breast_color::pink',
 'has_breast_color::purple',
 'has_breast_color::red',
 'has_breast_color::rufous',
 'has_breast_color::white',
 'has_breast_color::yellow',
 'has_breast_pattern::multi-colored',
 'has_breast_pattern::solid',
 'has_breast_pattern::spotted',
 'has_breast_pattern::striped',
 'has_crown_color::black',
 'has_crown_color::blue',
 'has_crown_color::brown',
 'has_crown_color::buff',
 'has_crown_color::green',
 'has_crown_color::grey',
 'has_crown_color::iridescent',
 'has_crown_color::olive',
 'has_crown_color::orange',
 'has_crown_color::pink',
 'has_crown_color::purple',
 'has_crown_color::red',
 'has_crown_color::rufous',
 'has_crown_color::white',
 'has_crown_color::yellow',
 'has_eye_color::black',
 'has_eye_color::blue',
 'has_eye_color::brown',
 'has_eye_color::buff',
 'has_eye_color::green',
 'has_eye_color::grey',
 'has_eye_color::olive',
 'has_eye_color::orange',
 'has_eye_color::pink',
 'has_eye_color::purple',
 'has_eye_color::red',
 'has_eye_color::rufous',
 'has_eye_color::white',
 'has_eye_color::yellow',
 'has_forehead_color::black',
 'has_forehead_color::blue',
 'has_forehead_color::brown',
 'has_forehead_color::buff',
 'has_forehead_color::green',
 'has_forehead_color::grey',
 'has_forehead_color::iridescent',
 'has_forehead_color::olive',
 'has_forehead_color::orange',
 'has_forehead_color::pink',
 'has_forehead_color::purple',
 'has_forehead_color::red',
 'has_forehead_color::rufous',
 'has_forehead_color::white',
 'has_forehead_color::yellow',
 'has_head_pattern::capped',
 'has_head_pattern::crested',
 'has_head_pattern::eyebrow',
 'has_head_pattern::eyeline',
 'has_head_pattern::eyering',
 'has_head_pattern::malar',
 'has_head_pattern::masked',
 'has_head_pattern::plain',
 'has_head_pattern::spotted',
 'has_head_pattern::striped',
 'has_head_pattern::unique_pattern',
 'has_leg_color::black',
 'has_leg_color::blue',
 'has_leg_color::brown',
 'has_leg_color::buff',
 'has_leg_color::green',
 'has_leg_color::grey',
 'has_leg_color::iridescent',
 'has_leg_color::olive',
 'has_leg_color::orange',
 'has_leg_color::pink',
 'has_leg_color::purple',
 'has_leg_color::red',
 'has_leg_color::rufous',
 'has_leg_color::white',
 'has_leg_color::yellow',
 'has_nape_color::black',
 'has_nape_color::blue',
 'has_nape_color::brown',
 'has_nape_color::buff',
 'has_nape_color::green',
 'has_nape_color::grey',
 'has_nape_color::iridescent',
 'has_nape_color::olive',
 'has_nape_color::orange',
 'has_nape_color::pink',
 'has_nape_color::purple',
 'has_nape_color::red',
 'has_nape_color::rufous',
 'has_nape_color::white',
 'has_nape_color::yellow',
 'has_primary_color::black',
 'has_primary_color::blue',
 'has_primary_color::brown',
 'has_primary_color::buff',
 'has_primary_color::green',
 'has_primary_color::grey',
 'has_primary_color::iridescent',
 'has_primary_color::olive',
 'has_primary_color::orange',
 'has_primary_color::pink',
 'has_primary_color::purple',
 'has_primary_color::red',
 'has_primary_color::rufous',
 'has_primary_color::white',
 'has_primary_color::yellow',
 'has_shape::chicken-like-marsh',
 'has_shape::duck-like',
 'has_shape::gull-like',
 'has_shape::hawk-like',
 'has_shape::hummingbird-like',
 'has_shape::long-legged-like',
 'has_shape::owl-like',
 'has_shape::perching-like',
 'has_shape::pigeon-like',
 'has_shape::sandpiper-like',
 'has_shape::swallow-like',
 'has_shape::tree-clinging-like',
 'has_shape::upland-ground-like',
 'has_shape::upright-perching_water-like',
 'has_size::large_(16_-_32_in)',
 'has_size::medium_(9_-_16_in)',
 'has_size::small_(5_-_9_in)',
 'has_size::very_large_(32_-_72_in)',
 'has_size::very_small_(3_-_5_in)',
 'has_tail_pattern::multi-colored',
 'has_tail_pattern::solid',
 'has_tail_pattern::spotted',
 'has_tail_pattern::striped',
 'has_tail_shape::fan-shaped_tail',
 'has_tail_shape::forked_tail',
 'has_tail_shape::notched_tail',
 'has_tail_shape::pointed_tail',
 'has_tail_shape::rounded_tail',
 'has_tail_shape::squared_tail',
 'has_throat_color::black',
 'has_throat_color::blue',
 'has_throat_color::brown',
 'has_throat_color::buff',
 'has_throat_color::green',
 'has_throat_color::grey',
 'has_throat_color::iridescent',
 'has_throat_color::olive',
 'has_throat_color::orange',
 'has_throat_color::pink',
 'has_throat_color::purple',
 'has_throat_color::red',
 'has_throat_color::rufous',
 'has_throat_color::white',
 'has_throat_color::yellow',
 'has_under_tail_color::black',
 'has_under_tail_color::blue',
 'has_under_tail_color::brown',
 'has_under_tail_color::buff',
 'has_under_tail_color::green',
 'has_under_tail_color::grey',
 'has_under_tail_color::iridescent',
 'has_under_tail_color::olive',
 'has_under_tail_color::orange',
 'has_under_tail_color::pink',
 'has_under_tail_color::purple',
 'has_under_tail_color::red',
 'has_under_tail_color::rufous',
 'has_under_tail_color::white',
 'has_under_tail_color::yellow',
 'has_underparts_color::black',
 'has_underparts_color::blue',
 'has_underparts_color::brown',
 'has_underparts_color::buff',
 'has_underparts_color::green',
 'has_underparts_color::grey',
 'has_underparts_color::iridescent',
 'has_underparts_color::olive',
 'has_underparts_color::orange',
 'has_underparts_color::pink',
 'has_underparts_color::purple',
 'has_underparts_color::red',
 'has_underparts_color::rufous',
 'has_underparts_color::white',
 'has_underparts_color::yellow',
 'has_upper_tail_color::black',
 'has_upper_tail_color::blue',
 'has_upper_tail_color::brown',
 'has_upper_tail_color::buff',
 'has_upper_tail_color::green',
 'has_upper_tail_color::grey',
 'has_upper_tail_color::iridescent',
 'has_upper_tail_color::olive',
 'has_upper_tail_color::orange',
 'has_upper_tail_color::pink',
 'has_upper_tail_color::purple',
 'has_upper_tail_color::red',
 'has_upper_tail_color::rufous',
 'has_upper_tail_color::white',
 'has_upper_tail_color::yellow',
 'has_upperparts_color::black',
 'has_upperparts_color::blue',
 'has_upperparts_color::brown',
 'has_upperparts_color::buff',
 'has_upperparts_color::green',
 'has_upperparts_color::grey',
 'has_upperparts_color::iridescent',
 'has_upperparts_color::olive',
 'has_upperparts_color::orange',
 'has_upperparts_color::pink',
 'has_upperparts_color::purple',
 'has_upperparts_color::red',
 'has_upperparts_color::rufous',
 'has_upperparts_color::white',
 'has_upperparts_color::yellow',
 'has_wing_color::black',
 'has_wing_color::blue',
 'has_wing_color::brown',
 'has_wing_color::buff',
 'has_wing_color::green',
 'has_wing_color::grey',
 'has_wing_color::iridescent',
 'has_wing_color::olive',
 'has_wing_color::orange',
 'has_wing_color::pink',
 'has_wing_color::purple',
 'has_wing_color::red',
 'has_wing_color::rufous',
 'has_wing_color::white',
 'has_wing_color::yellow',
 'has_wing_pattern::multi-colored',
 'has_wing_pattern::solid',
 'has_wing_pattern::spotted',
 'has_wing_pattern::striped',
 'has_wing_shape::broad-wings',
 'has_wing_shape::long-wings',
 'has_wing_shape::pointed-wings',
 'has_wing_shape::rounded-wings',
 'has_wing_shape::tapered-wings']
} # contains a list of the concept columns present in the dataset

# we need to create the concepts.csv file that contains: file_name, concepts (where each concept is a column and is prefixed with has_)
# so for datasets that have concepts with multiple values, e.g. "pigmentation" -> absent, diffuse irregular, ... we need to create subconcepts,
# like: has_pigmentation_absent, has_pigmentation_diffuse_irregular, ... --> this is NOT equal to concept_columns!
# and then indicate by 1 and 0 whether a concept is present in the image or not
# for the file_name column we need to add the split directory, so that it reads train/filename (or test/filename or val/filename)
# then the concepts.csv shall be saved in the datasets root

for dataset in datasets:
    combined_dfs = []
    
    # 1. Load and combine data from all splits
    for split in splits:
        path = f"/home/voigt/data/{dataset}/imgs/{split}/{meta_file}"
        if not os.path.exists(path):
            continue
            
        df = pd.read_csv(path)
        
        # Isolate target columns and format the filename to include the split
        fname_col = filename_column[dataset]
        target_cols = [fname_col] + concept_columns[dataset]
        df = df[target_cols].copy()
        
        df.rename(columns={fname_col: 'file_name'}, inplace=True)
        df['file_name'] = f"{split}/" + df['file_name'].astype(str)
        
        combined_dfs.append(df)
        
    if not combined_dfs:
        continue
        
    full_df = pd.concat(combined_dfs, ignore_index=True)
    
    # 2. Process concepts into binary vectors prefixed with `has_`
    final_columns = ['file_name']
    
    for concept in concept_columns[dataset]:
        if concept.startswith("has_"):
            # Concepts are already binarized/boolean (e.g., fitzpatrick/skincon, CUB_200_2011)
            # Ensure integer format (0/1)
            full_df[concept] = full_df[concept].astype(int)
            final_columns.append(concept)
        else:
            # Concepts are categorical and require one-hot encoding (e.g., 7ptderm)
            dummies = pd.get_dummies(full_df[concept], prefix=f"has_{concept}")
            
            # Format subconcept column names to replace spaces with underscores and convert to lowercase
            dummies.columns = [col.replace(' ', '_').lower() for col in dummies.columns]
            
            # Cast booleans to 0/1 integers
            dummies = dummies.astype(int)
            
            full_df = pd.concat([full_df, dummies], axis=1)
            final_columns.extend(dummies.columns.tolist())
    
    # 3. Filter only the file_name and generated subconcepts
    concepts_df = full_df[final_columns]
    
    # 4. Save to dataset root
    out_path = f"/home/voigt/data/{dataset}/concepts.csv"
    concepts_df.to_csv(out_path, index=False)


In [22]:

dataset = "fitzpatrick/skincon"
split = "train"
meta_file = "metadata.csv"
path = f"/home/voigt/data/{dataset}/imgs/{split}/{meta_file}"
#path = f"/home/voigt/data/{dataset}/{meta_file}"
df = pd.read_csv(path)

df.head()

,has_Vesicle,has_Papule,has_Macule,has_Plaque,has_Abscess,has_Pustule,has_Bulla,has_Patch,has_Nodule,has_Ulcer,...,has_Pigmented,has_Cyst,filepath,diag,partition,fitzpatrick,label,nine_partition_label,three_partition_label,file_name
0,0,0,0,1,0,0,0,0,0,0,...,0,0,ba-ce-ca_468/ba-ce-ca_f3_218_bb3d0878.jpg,ba-ce-ca,train,3,basal cell carcinoma,malignant epidermal,malignant,basal_cell_carcinoma/ba-ce-ca_f3_218_bb3d0878.jpg
1,0,1,0,1,0,0,0,0,0,0,...,0,0,al-co-de_430/al-co-de_f2_240_4cb78451.jpg,al-co-de,train,2,allergic contact dermatitis,inflammatory,non-neoplastic,allergic_contact_dermatitis/al-co-de_f2_240_4c...
2,0,0,0,0,0,0,0,1,0,0,...,0,0,lu-er_410/lu-er_f1_134_a82ebeac.jpg,lu-er,train,1,lupus erythematosus,inflammatory,non-neoplastic,lupus_erythematosus/lu-er_f1_134_a82ebeac.jpg
3,0,0,0,1,0,0,0,0,0,0,...,0,0,pi-ro_193/pi-ro_f1_5_f70e72f6.jpg,pi-ro,train,1,pityriasis rosea,inflammatory,non-neoplastic,pityriasis_rosea/pi-ro_f1_5_f70e72f6.jpg
4,0,0,0,1,0,0,0,0,0,0,...,0,0,sq-ce-ca_581/sq-ce-ca_f2_361_4953e564.jpg,sq-ce-ca,test,2,squamous cell carcinoma,malignant epidermal,malignant,squamous_cell_carcinoma/sq-ce-ca_f2_361_4953e5...


In [23]:
c = []
for col in df.columns: 
    if not col.startswith("has_"):
        c.append(col)
c

['filepath',
 'diag',
 'partition',
 'fitzpatrick',
 'label',
 'nine_partition_label',
 'three_partition_label',
 'file_name']

In [28]:
import pandas as pd
import os
dataset = "7ptderm"

for split in ["train", "val", "test"]:

    path = f"/home/voigt/data/{dataset}/imgs/{split}/metadata.csv"
    if not os.path.exists(path):
            continue
    
    df = pd.read_csv(path)
    df['disease'] = df['filepath'].str.split('/').str[0]
    df.to_csv(path, index=False)


In [2]:
import pandas as pd
dataset = "7ptderm"
split = "train"
path = f"/home/voigt/data/{dataset}/imgs/{split}/metadata.csv"

    
df = pd.read_csv(path)
df.head()

,file_name,old_label,seven_point_score,pigment_network,streaks,pigmentation,regression_structures,dots_and_globules,blue_whitish_veil,vascular_structures,...,management,clinic,image_path,image_type,label,malignancy,disease_label,CLIP_caption,filepath,disease
0,basal_cell_carcinoma/Nel026.jpg,basal cell carcinoma,0,absent,absent,absent,absent,absent,absent,arborizing,...,excision,NEL/NEL025.JPG,NEL/Nel026.jpg,dermoscopy,BCC,Malignant,basal cell carcinoma,Texture and color details of basal cell carcin...,basal_cell_carcinoma/Nel026.jpg,basal_cell_carcinoma
1,basal_cell_carcinoma/Nel033.jpg,basal cell carcinoma,1,absent,absent,absent,absent,irregular,absent,arborizing,...,excision,NEL/Nel032.jpg,NEL/Nel033.jpg,dermoscopy,BCC,Malignant,basal cell carcinoma,basal cell carcinoma (malignant).,basal_cell_carcinoma/Nel033.jpg,basal_cell_carcinoma
2,basal_cell_carcinoma/Nel037.jpg,basal cell carcinoma,1,absent,absent,diffuse irregular,absent,absent,absent,absent,...,excision,NEL/NEL036.JPG,NEL/Nel037.jpg,dermoscopy,BCC,Malignant,basal cell carcinoma,A medical image showing a malignant basal cell...,basal_cell_carcinoma/Nel037.jpg,basal_cell_carcinoma
3,basal_cell_carcinoma/Nel085.jpg,basal cell carcinoma,4,absent,absent,diffuse irregular,absent,irregular,present,absent,...,excision,NEL/Nel084.jpg,NEL/Nel085.jpg,dermoscopy,BCC,Malignant,basal cell carcinoma,A medical image showing a malignant basal cell...,basal_cell_carcinoma/Nel085.jpg,basal_cell_carcinoma
4,basal_cell_carcinoma/Nel089.jpg,basal cell carcinoma,2,absent,absent,diffuse irregular,absent,irregular,absent,arborizing,...,excision,NEL/NEL088.JPG,NEL/Nel089.jpg,dermoscopy,BCC,Malignant,basal cell carcinoma,Skin lesion: basal cell carcinoma.,basal_cell_carcinoma/Nel089.jpg,basal_cell_carcinoma


In [30]:
df["disease"].unique()

<ArrowStringArray>
[        'basal_cell_carcinoma',                   'blue_nevus',
                  'clark_nevus',               'combined_nevus',
             'congenital_nevus',                 'dermal_nevus',
               'dermatofibroma',                      'lentigo',
           'melanoma_(in_situ)', 'melanoma_(less_than_0.76_mm)',
    'melanoma_(0.76_to_1.5_mm)',  'melanoma_(more_than_1.5_mm)',
          'melanoma_metastasis',                    'melanosis',
                'miscellaneous',              'recurrent_nevus',
          'reed_or_spitz_nevus',         'seborrheic_keratosis',
              'vascular_lesion']
Length: 19, dtype: str

In [26]:
from vitlab.eval import load_concepts
from vitlab.datasets import get_dataset

DATASET = "7ptderm"

ds = get_dataset(name=DATASET)
concepts = load_concepts(dataset_name=DATASET)

Casting the dataset:   0%|          | 0/826 [00:00<?, ? examples/s]


ArrowInvalid: Failed to parse string: 'BCC' as a scalar of type int64

In [9]:
ds["train"]

Dataset({
    features: ['image', 'old_label', 'seven_point_score', 'pigment_network', 'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules', 'blue_whitish_veil', 'vascular_structures', 'elevation', 'location', 'sex', 'management', 'clinic', 'image_path', 'image_type', 'label', 'malignancy', 'disease_label', 'CLIP_caption'],
    num_rows: 826
})

In [3]:
concepts.image_ids

['train/001.Black_footed_Albatross/Black_Footed_Albatross_0009_34.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0074_59.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0014_89.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0031_100.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0051_796103.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0010_796097.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0023_796059.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0040_796066.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0089_796069.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0067_170.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0060_796076.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0056_796078.jpg',
 'train/001.Black_footed_Albatross/Black_Footed_Albatross_0080_796096.jpg',
 'train/001.Black_footed_Albat

In [11]:
grades = {
    "aud": [1.0, 6.0],
    "eiaps": [1.3, 6.0],
    "mfi": [2.3, 6.0],
    "kinf": [1.2, 9.0],
    "dbs": [1.0, 6.0],
    "eirbs": [2.0, 6.0],
    "fse": [2.0, 6.0],
    "wima": [1.0, 6.0],
    "eth": [1.0, 6.0],
    "convai": [1.0, 6.0],
    "ds": [1.7, 6.0],
    "deepl": [1.0, 6.0],
    "sem": [1.0, 3.0],
    "proj1": [1.0, 6.0],
    "proj2": [1.0, 6.0],
    "ma": [1.0, 30.0]
}

final = 0
for k, v in grades.items():
    final += v[0]*v[1]

print(final / 120)

1.2300000000000002


In [ ]:
import pandas as p